# B8 — Encoder quantization test (`quant_int8`)

**Question:** if MobileCLIP-S1's *weights* are quantized to int8 (the
change that actually saves app size: ~43 MB -> ~21 MB), what happens to
the adapter - and does **refitting W** (seconds, closed form) recover it?

**Method:** weight-only fake-quantization - every Conv2d/Linear weight
symmetric per-output-channel int8, dequantized in place - so the run
executes on GPU while numerically matching int8 weight storage. Then a
mini-B1/B2/B3 on 2,000 fresh COCO images (1,500 refit / 500 eval):

| variant | shows |
|---|---|
| fp encoder + original W | reference |
| **int8 encoder + original W** | the silent degradation |
| **int8 encoder + refit W** | the recovery |

**Pass criterion:** refit variant recovers to within 2 points of the
reference at every K. Runtime ~6-8 min GPU. Needs the COCO cache +
`adapter.npz`.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
!pip -q install open_clip_torch transformers pillow

In [ ]:
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ['DATA_DIR'])

def l2n(X):
    return X / (np.linalg.norm(X, axis=-1, keepdims=True) + 1e-9)

def recall(sim, ks=(1, 5, 10)):
    ranks = (-sim).argsort(axis=1)
    n = sim.shape[0]
    return {k: float((ranks[:, :k] == np.arange(n)[:, None]).any(1).mean())
            for k in ks}

def report(name, r, ceil=None):
    line = f"{name:<38} " + "  ".join(f"R@{k}={r[k]:.3f}" for k in (1, 5, 10))
    if ceil:
        pct = min(100 * r[k] / max(ceil[k], 1e-9) for k in (1, 5, 10))
        line += f"   (worst-K {pct:.1f}% of ref)"
    print(line)

import torch, json, copy
from PIL import Image
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

import open_clip
mob, _, mob_pre = open_clip.create_model_and_transforms(
    'MobileCLIP-S1', pretrained='datacompdr')
mob.eval().to(DEV)

def fake_quant_int8_(model):
    """Weight-only symmetric per-output-channel int8 quant-dequant,
    in place. Numerically equivalent to storing weights as int8+scale."""
    n = 0
    with torch.no_grad():
        for m in model.modules():
            if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear)):
                w = m.weight.data
                flat = w.reshape(w.shape[0], -1)
                scale = flat.abs().amax(dim=1).clamp_min(1e-8) / 127.0
                q = torch.round(flat / scale[:, None]).clamp(-127, 127)
                m.weight.data = (q * scale[:, None]).reshape(w.shape)
                n += 1
    return n

mob_q = copy.deepcopy(mob)
n = fake_quant_int8_(mob_q)
print(f'quantized {n} Conv2d/Linear weight tensors to int8 (dequantized view)')

from transformers import AutoModel, AutoProcessor
SIG = 'google/siglip2-base-patch16-224'
sig = AutoModel.from_pretrained(SIG).eval().to(DEV)
sig_proc = AutoProcessor.from_pretrained(SIG)

In [ ]:
# Mini-B1: 2,000 fresh COCO pairs through fp encoder, int8 encoder, SigLIP
img_dir = DATA_DIR / 'coco' / 'val2017'
ann = json.load(open(DATA_DIR / 'coco' / 'annotations' /
                     'captions_val2017.json'))
id2file = {im['id']: im['file_name'] for im in ann['images']}
id2cap = {}
for a in ann['annotations']:
    id2cap.setdefault(a['image_id'], a['caption'])
ids = [i for i in id2cap if (img_dir / id2file[i]).exists()][:2000]

@torch.no_grad()
def encode_all(ids, bs=64):
    m_fp, m_q, s_img, s_txt = [], [], [], []
    for i in range(0, len(ids), bs):
        chunk = ids[i:i+bs]
        imgs = [Image.open(img_dir / id2file[j]).convert('RGB')
                for j in chunk]
        caps = [id2cap[j] for j in chunk]
        b = torch.stack([mob_pre(im) for im in imgs]).to(DEV)
        m_fp.append(torch.nn.functional.normalize(
            mob.encode_image(b), dim=-1).float().cpu().numpy())
        m_q.append(torch.nn.functional.normalize(
            mob_q.encode_image(b), dim=-1).float().cpu().numpy())
        x = sig_proc(images=imgs, return_tensors='pt').to(DEV)
        e = sig.get_image_features(**x)
        e = e if torch.is_tensor(e) else e.pooler_output
        s_img.append(l2n(e.float().cpu().numpy()))
        t = sig_proc(text=caps, return_tensors='pt', padding='max_length',
                     truncation=True, max_length=64).to(DEV)
        e = sig.get_text_features(**t)
        e = e if torch.is_tensor(e) else e.pooler_output
        s_txt.append(l2n(e.float().cpu().numpy()))
        if (i//bs) % 10 == 0:
            print(f'  {i}/{len(ids)}')
    return (np.concatenate(m_fp), np.concatenate(m_q),
            np.concatenate(s_img), np.concatenate(s_txt))

mob_fp, mob_int8, sig_img, sig_txt = encode_all(ids)
drift = (mob_fp * mob_int8).sum(1)
print(f'\nencoder drift from int8 weights: mean cosine '
      f'{drift.mean():.4f}, min {drift.min():.4f}')

In [ ]:
# Refit W on int8-encoder embeddings (closed form, seconds) and evaluate
tr, te = np.arange(1500), np.arange(1500, 2000)
W_orig = np.load(DATA_DIR / 'adapter.npz')['W_ridge'].astype(np.float32)

def ridge(X, Y, a=1e-2):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + a*np.eye(d), X.T @ Y)

W_refit = ridge(mob_int8[tr], sig_img[tr])

txt, sig_g = sig_txt[te], sig_img[te]
ceil = recall(txt @ sig_g.T)
r_ref  = recall(txt @ l2n(mob_fp[te]  @ W_orig).T)
r_deg  = recall(txt @ l2n(mob_int8[te] @ W_orig).T)
r_fix  = recall(txt @ l2n(mob_int8[te] @ W_refit).T)
report('SigLIP native (ceiling)', ceil)
report('fp encoder + original W (reference)', r_ref, ceil)
report('int8 encoder + ORIGINAL W (degraded?)', r_deg, ceil)
report('int8 encoder + REFIT W (recovered?)', r_fix, ceil)
ok = all(r_fix[k] >= r_ref[k] - 0.02 for k in (1, 5, 10))
print('\nPASS: refit recovers within 2pts of reference' if ok
      else 'CHECK: refit did not fully recover - int8 encoder costs real '
           'accuracy, weigh against the 21 MB saving')

**Interpretation guide.** Three outcomes are informative: if the
*degraded* row barely moves, int8 weights are nearly free and refitting
is optional insurance. If degraded drops but *refit* recovers, the
report's rule is demonstrated live: quantize -> refit -> revalidate, in
that order. If even refit falls short, the encoder's int8 loss is real
information loss (same logic as the MLP experiment: no map recovers what
the encoder no longer emits). Note this is *weight-only* quantization -
full int8 activation quantization on the ANE can differ; treat a pass
here as necessary, not sufficient, and confirm on-device.